# Support Vector Machine

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Importing the dataset

In [3]:
dataset = pd.read_csv("../Data/bank.csv", sep = ";")
dataset

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,329,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,153,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,151,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,129,4,211,3,other,no


In [4]:
X = dataset.iloc[:, :-1].copy()
y = dataset.iloc[:, -1].copy()
X, y

(      age            job  marital  education default  balance housing loan  \
 0      30     unemployed  married    primary      no     1787      no   no   
 1      33       services  married  secondary      no     4789     yes  yes   
 2      35     management   single   tertiary      no     1350     yes   no   
 3      30     management  married   tertiary      no     1476     yes  yes   
 4      59    blue-collar  married  secondary      no        0     yes   no   
 ...   ...            ...      ...        ...     ...      ...     ...  ...   
 4516   33       services  married  secondary      no     -333     yes   no   
 4517   57  self-employed  married   tertiary     yes    -3313     yes  yes   
 4518   57     technician  married  secondary      no      295      no   no   
 4519   28    blue-collar  married  secondary      no     1137      no   no   
 4520   44   entrepreneur   single   tertiary      no     1136     yes  yes   
 
        contact  day month  duration  campaign  pd

## Binary Categorical Column

In [5]:
binaryCols = [
    col for col in X.columns
    if X[col].dtype == 'object' and X[col].nunique() == 2
]

print(binaryCols)

['default', 'housing', 'loan']


## Multi-category Categorical Columns

In [6]:
multiCatCols = [
    col for col in X.columns
    if X[col].dtype == 'object' and X[col].nunique() > 2
]

print(multiCatCols)

['job', 'marital', 'education', 'contact', 'month', 'poutcome']


## Numerical Columns

In [7]:
numericalCols = [
    col for col in X.columns
    if X[col].dtype != 'object'
]

print(numericalCols)

['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


## Label Encode Binary Columns

In [8]:
from sklearn.preprocessing import LabelEncoder

for col in binaryCols:
    X[col] = LabelEncoder().fit_transform(X[col])

## Label Encode the Target

In [9]:
y = LabelEncoder().fit_transform(y)

## Train-Test Split

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=0
)

## Split Train data into Train & Val set

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=0
)

## Encoding multi-category categorical columns

OneHot Encoding is done after `train_test_split` because OneHotEncoder learns from the data.

Anything that learns from the data should be fit on training data only.

This avoids data leakage.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        (
            'encoder',
            OneHotEncoder(handle_unknown='ignore'),
            multiCatCols
        )
    ],
    
    remainder='passthrough'
)

In [13]:
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)
X_val = ct.transform(X_val)

In [14]:
X_train, X_test, X_val

(array([[ 0.,  0.,  0., ...,  1., -1.,  0.],
        [ 0.,  0.,  0., ...,  3., -1.,  0.],
        [ 1.,  0.,  0., ...,  1., -1.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  2., -1.,  0.],
        [ 0.,  0.,  0., ...,  2., -1.,  0.],
        [ 0.,  0.,  0., ...,  2., -1.,  0.]], shape=(2892, 48)),
 array([[  0.,   0.,   0., ...,   2.,  -1.,   0.],
        [  0.,   0.,   0., ...,   1., 100.,   2.],
        [  0.,   1.,   0., ...,   7.,  -1.,   0.],
        ...,
        [  0.,   0.,   0., ...,   2.,  -1.,   0.],
        [  0.,   0.,   0., ...,   7.,  -1.,   0.],
        [  0.,   0.,   1., ...,   2.,  -1.,   0.]], shape=(905, 48)),
 array([[  0.,   0.,   0., ...,   1., 343.,   2.],
        [  0.,   0.,   0., ...,   6.,  -1.,   0.],
        [  0.,   0.,   0., ...,   1.,  -1.,   0.],
        ...,
        [  0.,   1.,   0., ...,   2., 178.,   5.],
        [  0.,   1.,   0., ...,   5.,  -1.,   0.],
        [  0.,   0.,   0., ...,   1.,  -1.,   0.]], shape=(724, 48)))

In [15]:
feature_names = ct.get_feature_names_out()

for i, feature in enumerate(feature_names):
    print(i, feature)

0 encoder__job_admin.
1 encoder__job_blue-collar
2 encoder__job_entrepreneur
3 encoder__job_housemaid
4 encoder__job_management
5 encoder__job_retired
6 encoder__job_self-employed
7 encoder__job_services
8 encoder__job_student
9 encoder__job_technician
10 encoder__job_unemployed
11 encoder__job_unknown
12 encoder__marital_divorced
13 encoder__marital_married
14 encoder__marital_single
15 encoder__education_primary
16 encoder__education_secondary
17 encoder__education_tertiary
18 encoder__education_unknown
19 encoder__contact_cellular
20 encoder__contact_telephone
21 encoder__contact_unknown
22 encoder__month_apr
23 encoder__month_aug
24 encoder__month_dec
25 encoder__month_feb
26 encoder__month_jan
27 encoder__month_jul
28 encoder__month_jun
29 encoder__month_mar
30 encoder__month_may
31 encoder__month_nov
32 encoder__month_oct
33 encoder__month_sep
34 encoder__poutcome_failure
35 encoder__poutcome_other
36 encoder__poutcome_success
37 encoder__poutcome_unknown
38 remainder__age
39 rem

In [16]:
print(X_train, X_test, X_val)

[[ 0.  0.  0. ...  1. -1.  0.]
 [ 0.  0.  0. ...  3. -1.  0.]
 [ 1.  0.  0. ...  1. -1.  0.]
 ...
 [ 0.  0.  0. ...  2. -1.  0.]
 [ 0.  0.  0. ...  2. -1.  0.]
 [ 0.  0.  0. ...  2. -1.  0.]] [[  0.   0.   0. ...   2.  -1.   0.]
 [  0.   0.   0. ...   1. 100.   2.]
 [  0.   1.   0. ...   7.  -1.   0.]
 ...
 [  0.   0.   0. ...   2.  -1.   0.]
 [  0.   0.   0. ...   7.  -1.   0.]
 [  0.   0.   1. ...   2.  -1.   0.]] [[  0.   0.   0. ...   1. 343.   2.]
 [  0.   0.   0. ...   6.  -1.   0.]
 [  0.   0.   0. ...   1.  -1.   0.]
 ...
 [  0.   1.   0. ...   2. 178.   5.]
 [  0.   1.   0. ...   5.  -1.   0.]
 [  0.   0.   0. ...   1.  -1.   0.]]


## Identify all the Numerical columns

In [17]:
numericalIndices = [
    i
    for i, feature in enumerate(feature_names)
    if feature.startswith("remainder__")
    and not set(X_train[:, i]).issubset({0, 1})
]

print(numericalIndices)

[38, 40, 43, 44, 45, 46, 47]


## Feature Scale the Numerical Columns

In [18]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train[:, numericalIndices] = sc.fit_transform(
    X_train[:, numericalIndices]
)

X_test[:, numericalIndices] = sc.transform(
    X_test[:, numericalIndices]
)

X_val[:, numericalIndices] = sc.transform(
    X_val[:, numericalIndices]
)

In [19]:
X_train, X_test, X_val

(array([[ 0.        ,  0.        ,  0.        , ..., -0.58882345,
         -0.40174297, -0.31084491],
        [ 0.        ,  0.        ,  0.        , ...,  0.07857903,
         -0.40174297, -0.31084491],
        [ 1.        ,  0.        ,  0.        , ..., -0.58882345,
         -0.40174297, -0.31084491],
        ...,
        [ 0.        ,  0.        ,  0.        , ..., -0.25512221,
         -0.40174297, -0.31084491],
        [ 0.        ,  0.        ,  0.        , ..., -0.25512221,
         -0.40174297, -0.31084491],
        [ 0.        ,  0.        ,  0.        , ..., -0.25512221,
         -0.40174297, -0.31084491]], shape=(2892, 48)),
 array([[ 0.        ,  0.        ,  0.        , ..., -0.25512221,
         -0.40174297, -0.31084491],
        [ 0.        ,  0.        ,  0.        , ..., -0.58882345,
          0.60817069,  0.85739873],
        [ 0.        ,  1.        ,  0.        , ...,  1.41338398,
         -0.40174297, -0.31084491],
        ...,
        [ 0.        ,  0.        ,  

In [20]:
y_train, y_test, y_val

(array([0, 0, 0, ..., 0, 0, 0], shape=(2892,)),
 array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,

## Oversampling

In [21]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)

X_train, y_train = smote.fit_resample(
    X_train,
    y_train
)

## Train the Random Forest Model

In [22]:
from sklearn.svm import SVC
from sklearn.metrics import f1_score

bestF1 = 0
bestModel = None

for C in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    for gamma in ['scale', 'auto', 0.0001, 0.001, 0.01, 0.1, 1]:
        model = SVC(
            kernel='rbf',
            C=C,
            gamma=gamma,
            class_weight='balanced',
            random_state=0
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        score = f1_score(y_val, y_pred)

        if score > bestF1:
            bestF1 = score
            bestModel = model

print("Best F1:", bestF1)
print("Best Model:", bestModel)

Best F1: 0.5277777777777778
Best Model: SVC(C=1, class_weight='balanced', gamma=0.1, random_state=0)


## Predicting the Test set Results

In [23]:
y_pred = bestModel.predict(X_test)

## Evaluate the Model

In [24]:
from sklearn.metrics import confusion_matrix, accuracy_score

acc = accuracy_score(y_test, y_pred) * 100
cm = confusion_matrix(y_test, y_pred)

print(acc)
print("\n")
print(cm)

88.61878453038673


[[732  69]
 [ 34  70]]


## Classification Report

In [25]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.91      0.93       801
           1       0.50      0.67      0.58       104

    accuracy                           0.89       905
   macro avg       0.73      0.79      0.76       905
weighted avg       0.90      0.89      0.89       905

